In [1]:
import numpy as np
import subprocess
import os
from pathlib import Path
import time
import xarray as xr

In [2]:
#Creates a folder that makes testing easier
def unproblematic_folder(dir):

    new_dir = dir

    count = 1
            
    while True:
    
        try: 
            os.mkdir(new_dir) #Create new directory
        except: #If it doesn't work because a directory of the same name already exists, 
                #check if that directory is empty
            
            with os.scandir(new_dir) as it: 
                
                if not any(it): #if it is empty break the loop and set it as the new directory
                    break
                else: #otherwise set a new directory name and then repeat the same process
                    count += 1
                    new_dir = f"{dir}_{str(count)}"

    new_dir = Path(new_dir)
    return new_dir
                

In [10]:
#Setting directories for output and log files
name = "test"
dir = unproblematic_folder(f"/glade/derecho/scratch/considine/{name}")
job_dir = unproblematic_folder(f"/glade/u/home/considine/{dir.name}")
    
month = 0

longitude = np.linspace(-180,172,45)
latitude = np.linspace(-80, 72, 20)
coordinates = []

#Creating a list of coordinate values and labels
for lat in latitude:

    lat = int(lat)

    for lon in longitude:

        lon = int(lon)
        lon2 = lon + 8
        lat2 = lat + 8
        
        if lat < 0:
            coord_name = f'nprecip_n{str(abs(lat)).zfill(3)}_'
        else:
            coord_name = f'nprecip_{str(lat).zfill(4)}_'
        if lon < 0:
            coord_name = f'{coord_name}n{str(abs(lon)).zfill(3)}.nc'
        else:
            coord_name = f'{coord_name}{str(lon).zfill(4)}.nc'

        coordinates.append([lat,lat2,lon,lon2, coord_name])

In [12]:
#Creating job script
casper_job = ['#!/bin/bash', 
             '#PBS -A WYOM0161', 
             '#PBS -l walltime=00:30:00', 
             '#PBS -q casper', 
             '#PBS -l select=1:ncpus=1:mem=16GB', 
             '#PBS -N ', 
             '#PBS -e ', 
             '#PBS -o ', 
             'module load conda', 
             'conda activate analysis', 
             'python normalize_precip.py latRange lonRange month dir'] 

#Submitting a job to casper to normalize the data at each coordinate
for c in coordinates:
    
    job = f"job_{c[4][8:17]}"
    out_dir = f"{dir}/{c[4]}"
    arg1 = str([c[0],c[1]]).replace(" ", "")
    arg2 = str([c[2],c[3]]).replace(" ", "")
    
    casper_job[5] = f'#PBS -N {job}'
    casper_job[6] = f'#PBS -e {job_dir}/{job}_e.txt'
    casper_job[7] = f'#PBS -o {job_dir}/{job}_o.txt'
    casper_job[10] = f'python normalize_precip.py {arg1} {arg2} {str(month)} {out_dir}'

    with open(f'{job}.sh',"w") as f:
        f.write('\n'.join(casper_job))

    subprocess.call(f'qsub {job}.sh',shell=True)

    time.sleep(1)
    os.remove(f'/glade/u/home/considine/{job}.sh')

5750385.casper-pbs
